# GPU Stats Analysis

This notebook combines performance metrics from hyperparameter search with GPU and timing statistics from rerun experiments.

**Data Sources:**
1. `best_configs_summary.csv` - Performance metrics (F1, accuracy, AUROC)
2. W&B `bgbench_best_configs_rerun` project - GPU memory and timing stats

In [ ]:
# Cell 1: Imports and Configuration
import pandas as pd
import numpy as np
import wandb
from pathlib import Path
import re
import warnings
warnings.filterwarnings('ignore')

# Configuration
WANDB_ENTITY = "bioshape-lab"
WANDB_PROJECT = "bgbench_best_configs_rerun"
SUMMARY_CSV = Path("../scripts/rerun_best_configs/best_configs_summary.csv")

# Model name mapping (CSV uses different naming conventions)
MODEL_NAME_MAP = {
    "sage": "graph_sage",
    "GATv4": "gatv4",
    "GATv2": "gatv2",
    # Others are already lowercase and match
}

# Metrics to fetch from W&B
WANDB_METRICS = [
    "GPU/peak_memory_allocated_GB",
    "GPU/peak_memory_reserved_GB",
    "AvgTime/train_epoch_mean",
    "AvgTime/train_epoch_std",
    "AvgTime/val_epoch_mean",
    "AvgTime/val_epoch_std",
]

print(f"Summary CSV exists: {SUMMARY_CSV.exists()}")

In [ ]:
# Cell 2: Fetch W&B Runs

def parse_run_name(run_name: str) -> dict | None:
    """Parse run name to extract dataset, model, method, ratio, and readout.
    
    Run names follow the pattern: {dataset}_{model}_{method}_{ratio}
    For OmicsReadOut runs, the readout info is in the tags.
    """
    # Pattern: dataset_model_method_ratio
    # Examples: addneuromed_mlp_correlation_0.3, parkinsons_GATv4_variance_0.5
    
    # Known datasets
    datasets = ["addneuromed", "parkinsons", "motrpac"]
    
    # Known models (include both original and normalized names)
    # Map to normalized lowercase names
    model_variants = {
        "mlp": "mlp",
        "gcn": "gcn", 
        "gin": "gin",
        "gatv2": "gatv2",
        "GATv2": "gatv2",
        "gatv4": "gatv4",
        "GATv4": "gatv4",
        "chebnet": "chebnet",
        "graph_sage": "graph_sage",
        "sage": "graph_sage",  # W&B uses 'sage' in run names
        "sagn": "sagn",
    }
    
    # Known methods (check longer ones first to avoid partial matches)
    methods = ["distance_correlation", "correlation", "variance", "random"]
    
    # Try to find dataset
    dataset = None
    for d in datasets:
        if run_name.startswith(d + "_"):
            dataset = d
            break
    
    if not dataset:
        return None
    
    # Remove dataset prefix
    remaining = run_name[len(dataset)+1:]  # +1 for the underscore
    
    # Try to find model (check all variants)
    model = None
    model_normalized = None
    for variant, normalized in model_variants.items():
        if remaining.startswith(variant + "_"):
            model = variant
            model_normalized = normalized
            break
    
    if not model:
        return None
    
    # Remove model prefix
    remaining = remaining[len(model)+1:]
    
    # Try to find method (check longer methods first)
    method = None
    for m in methods:
        if remaining.startswith(m + "_"):
            method = m
            break
    
    if not method:
        return None
    
    # Remove method prefix - the rest should be the ratio
    remaining = remaining[len(method)+1:]
    
    # Parse ratio (should be a float like 0.3, 0.5, 0.8, 1.0)
    try:
        ratio = float(remaining)
    except ValueError:
        return None
    
    return {
        "dataset": dataset,
        "model": model_normalized,  # Use normalized model name
        "method": method,
        "node_sample_ratio": ratio,
    }


def fetch_wandb_runs():
    """Fetch all runs from W&B and extract GPU/timing metrics."""
    from tqdm.auto import tqdm
    
    api = wandb.Api()
    
    # Get all runs from the project
    runs = api.runs(f"{WANDB_ENTITY}/{WANDB_PROJECT}")
    
    print(f"Found {len(runs)} runs in {WANDB_ENTITY}/{WANDB_PROJECT}")
    
    run_data = []
    skipped = 0
    for run in tqdm(runs, desc="Fetching runs"):
        # Skip failed or running runs
        if run.state != "finished":
            continue
        
        # Parse run name
        parsed = parse_run_name(run.name)
        if not parsed:
            skipped += 1
            continue
        
        # Get readout from tags
        tags = run.tags
        if "rerun_best" not in tags:
            continue  # Not a rerun job
        
        # Get readout from nested config: config['model']['readout']['readout_name']
        readout = "NoReadOut"  # Default
        if hasattr(run, 'config') and run.config:
            config = run.config
            # Try nested access
            model_cfg = config.get("model", {})
            if isinstance(model_cfg, dict):
                readout_cfg = model_cfg.get("readout", {})
                if isinstance(readout_cfg, dict):
                    readout = readout_cfg.get("readout_name", "NoReadOut")
        
        # Extract metrics from summary
        summary = run.summary
        metrics = {"run_name": run.name, "run_id": run.id, "readout": readout}
        metrics.update(parsed)
        
        for metric in WANDB_METRICS:
            metrics[metric] = summary.get(metric, None)
        
        run_data.append(metrics)
    
    print(f"Parsed {len(run_data)} valid runs (skipped {skipped} unparseable)")
    return pd.DataFrame(run_data)


# Fetch runs (this may take a minute)
print("Fetching W&B runs...")
wandb_df = fetch_wandb_runs()
print(f"\nFetched {len(wandb_df)} runs")

# Save wandb_df to CSV for faster loading next time
wandb_df_file = Path("../scripts/rerun_best_configs/wandb_runs_raw.csv")
wandb_df.to_csv(wandb_df_file, index=False)
print(f"Saved raw W&B runs to: {wandb_df_file}")

wandb_df.head()

In [ ]:
# Alternative: Load merged_df from saved CSV (skip W&B fetching)
# Uncomment and run this cell instead of cells 2-6 if you want to load from saved data

MERGED_CSV = Path("../scripts/rerun_best_configs/merged_performance_gpu_stats.csv")

if MERGED_CSV.exists():
    merged_df = pd.read_csv(MERGED_CSV)
    print(f"Loaded merged_df from {MERGED_CSV}: {len(merged_df)} rows")
    display(merged_df.head())
else:
    print(f"File not found: {MERGED_CSV}")
    print("Run cells 2-6 to fetch from W&B and create the merged data.")

In [ ]:
# Cell 3: Aggregate W&B runs by configuration (across seeds)

GROUP_COLS = ["dataset", "model", "method", "node_sample_ratio", "readout"]
METRIC_COLS = [
    "GPU/peak_memory_allocated_GB",
    "GPU/peak_memory_reserved_GB", 
    "AvgTime/train_epoch_mean",
    "AvgTime/train_epoch_std",
    "AvgTime/val_epoch_mean",
    "AvgTime/val_epoch_std",
]

def aggregate_runs(df):
    """Aggregate runs by configuration, computing mean and std across seeds."""
    
    # Group by configuration
    grouped = df.groupby(GROUP_COLS)
    
    # Compute mean and std for each metric
    agg_dict = {col: ["mean", "std", "count"] for col in METRIC_COLS}
    
    agg_df = grouped.agg(agg_dict)
    
    # Flatten column names
    agg_df.columns = [f"{col}_{stat}" for col, stat in agg_df.columns]
    
    # Reset index to make group columns regular columns
    agg_df = agg_df.reset_index()
    
    return agg_df


wandb_agg = aggregate_runs(wandb_df)
print(f"Aggregated to {len(wandb_agg)} unique configurations")
print(f"\nRuns per configuration (before filtering):")
print(wandb_agg["GPU/peak_memory_allocated_GB_count"].value_counts())

# Filter to only keep configurations with exactly 3 runs (3 seeds)
# REQUIRED_RUNS = 3
# wandb_agg_filtered = wandb_agg[wandb_agg["GPU/peak_memory_allocated_GB_count"] == REQUIRED_RUNS].copy()
# print(f"\nAfter filtering for exactly {REQUIRED_RUNS} runs: {len(wandb_agg_filtered)} configurations")

# # Use filtered data
# wandb_agg = wandb_agg_filtered
wandb_agg.head()

In [ ]:
# Cell 4: Load and Prepare Performance Data

# Load the summary CSV
perf_df = pd.read_csv(SUMMARY_CSV)
print(f"Loaded {len(perf_df)} configurations from {SUMMARY_CSV}")

# Normalize model names to match W&B run names
perf_df["model_normalized"] = perf_df["model"].replace(MODEL_NAME_MAP).str.lower()

# Show unique values
print("\nUnique values in performance data:")
print(f"  Models: {sorted(perf_df['model_normalized'].unique())}")
print(f"  Datasets: {sorted(perf_df['dataset'].unique())}")
print(f"  Methods: {sorted(perf_df['method'].unique())}")
print(f"  Ratios: {sorted(perf_df['node_sample_ratio'].unique())}")
print(f"  Readouts: {sorted(perf_df['readout'].unique())}")

# Check for trainable_params column
if "trainable_params" in perf_df.columns:
    print(f"\n  Trainable params range: {perf_df['trainable_params'].min():.0f} - {perf_df['trainable_params'].max():.0f}")
else:
    print("\n  Warning: trainable_params column not found. Re-run best_configs_analysis.ipynb to include it.")

perf_df.head()

In [ ]:
# Cell 5: Merge DataFrames

# Prepare performance data for merge
perf_merge = perf_df.copy()
perf_merge["model"] = perf_merge["model_normalized"]
perf_merge = perf_merge.drop(columns=["model_normalized"])

# Merge on group columns
merged_df = perf_merge.merge(
    wandb_agg,
    on=GROUP_COLS,
    how="left",
    indicator=True
)

# Check merge results
print("Merge results:")
print(merged_df["_merge"].value_counts())

# Show missing GPU stats (runs still in progress or failed)
missing = merged_df[merged_df["_merge"] == "left_only"]
if len(missing) > 0:
    print(f"\n{len(missing)} configurations missing GPU stats:")
    print(missing[GROUP_COLS].head(20))

# Clean up
merged_df = merged_df.drop(columns=["_merge"])

print(f"\nFinal merged dataset: {len(merged_df)} configurations")
merged_df.head()

In [ ]:
# Cell 6: Generate LaTeX Tables

def format_metric_with_std(mean_col, std_col, precision=3):
    """Format a metric with its standard deviation as 'mean ± std'."""
    def formatter(row):
        mean = row[mean_col]
        std = row[std_col]
        if pd.isna(mean):
            return "-"
        if pd.isna(std):
            return f"{mean:.{precision}f}"
        return f"{mean:.{precision}f} ± {std:.{precision}f}"
    return formatter


def create_summary_table(df, group_by="dataset"):
    """Create a summary table showing best configs per group."""
    
    # Select relevant columns
    table_cols = [
        "dataset", "model", "method", "node_sample_ratio", "readout",
        "summary.best_test/f1_macro", "summary.best_test/f1_macro_std",
        "GPU/peak_memory_allocated_GB_mean", "GPU/peak_memory_allocated_GB_std",
        "AvgTime/train_epoch_mean_mean", "AvgTime/train_epoch_mean_std",
        "AvgTime/val_epoch_mean_mean", "AvgTime/val_epoch_mean_std",
    ]
    
    # Filter to only rows with GPU stats
    table_df = df[df["GPU/peak_memory_allocated_GB_mean"].notna()].copy()
    
    if len(table_df) == 0:
        print("No data with GPU stats available yet!")
        return None
    
    # Create formatted columns
    table_df["F1 Macro"] = table_df.apply(
        format_metric_with_std("summary.best_test/f1_macro", "summary.best_test/f1_macro_std", 3), axis=1
    )
    table_df["GPU Memory (GB)"] = table_df.apply(
        format_metric_with_std("GPU/peak_memory_allocated_GB_mean", "GPU/peak_memory_allocated_GB_std", 2), axis=1
    )
    table_df["Train Time (s)"] = table_df.apply(
        format_metric_with_std("AvgTime/train_epoch_mean_mean", "AvgTime/train_epoch_mean_std", 2), axis=1
    )
    table_df["Val Time (s)"] = table_df.apply(
        format_metric_with_std("AvgTime/val_epoch_mean_mean", "AvgTime/val_epoch_mean_std", 2), axis=1
    )
    
    # Format trainable params if available
    if "trainable_params" in table_df.columns:
        def fmt_params(val):
            if pd.isna(val):
                return "-"
            if val >= 1e6:
                return f"{val/1e6:.1f}M"
            elif val >= 1e3:
                return f"{val/1e3:.0f}K"
            else:
                return f"{val:.0f}"
        table_df["Params"] = table_df["trainable_params"].apply(fmt_params)
    
    # Select display columns
    display_cols = ["dataset", "model", "method", "node_sample_ratio", "readout", 
                    "F1 Macro", "GPU Memory (GB)", "Train Time (s)", "Val Time (s)"]
    if "Params" in table_df.columns:
        display_cols.insert(5, "Params")  # Insert after readout
    
    return table_df[display_cols].sort_values(["dataset", "model", "method", "node_sample_ratio", "readout"])


# Create summary table
summary_table = create_summary_table(merged_df)
if summary_table is not None:
    print(f"Summary table: {len(summary_table)} configurations with GPU stats")
    summary_table.head(20)

In [ ]:
# Cell 7: Generate LaTeX Tables for Publication

def generate_latex_table(df, caption, label, columns=None):
    """Generate a LaTeX table from a DataFrame."""
    if df is None or len(df) == 0:
        return "% No data available"
    
    if columns:
        df = df[columns]
    
    # Rename columns for LaTeX
    col_rename = {
        "dataset": "Dataset",
        "model": "Model", 
        "method": "Method",
        "node_sample_ratio": "Ratio",
        "readout": "Readout",
    }
    df = df.rename(columns=col_rename)
    
    latex = df.to_latex(
        index=False,
        escape=False,  # Allow ± symbol
        caption=caption,
        label=label,
        column_format="l" * len(df.columns),
        position="htbp",
    )
    
    return latex


# Table 1: Best configuration per dataset (highest F1 Macro)
def get_best_per_dataset(df):
    """Get the best configuration per dataset based on F1 Macro."""
    if df is None or len(df) == 0:
        return None
    
    # Need to use original numeric columns for comparison
    df_with_f1 = df.copy()
    
    # Get best per dataset
    best_idx = merged_df.groupby("dataset")["summary.best_test/f1_macro"].idxmax()
    best_df = merged_df.loc[best_idx]
    
    # Add formatted columns
    best_df = best_df.copy()
    best_df["F1 Macro"] = best_df.apply(
        format_metric_with_std("summary.best_test/f1_macro", "summary.best_test/f1_macro_std", 3), axis=1
    )
    
    if "GPU/peak_memory_allocated_GB_mean" in best_df.columns:
        best_df["GPU Memory (GB)"] = best_df.apply(
            format_metric_with_std("GPU/peak_memory_allocated_GB_mean", "GPU/peak_memory_allocated_GB_std", 2), axis=1
        )
        best_df["Train Time (s)"] = best_df.apply(
            format_metric_with_std("AvgTime/train_epoch_mean_mean", "AvgTime/train_epoch_mean_std", 2), axis=1
        )
        best_df["Val Time (s)"] = best_df.apply(
            format_metric_with_std("AvgTime/val_epoch_mean_mean", "AvgTime/val_epoch_mean_std", 2), axis=1
        )
    
    # Format trainable params if available
    if "trainable_params" in best_df.columns:
        def fmt_params(val):
            if pd.isna(val):
                return "-"
            if val >= 1e6:
                return f"{val/1e6:.1f}M"
            elif val >= 1e3:
                return f"{val/1e3:.0f}K"
            else:
                return f"{val:.0f}"
        best_df["Params"] = best_df["trainable_params"].apply(fmt_params)
    
    display_cols = ["dataset", "model", "method", "node_sample_ratio", "readout", "F1 Macro"]
    if "Params" in best_df.columns:
        display_cols.append("Params")
    if "GPU Memory (GB)" in best_df.columns:
        display_cols.extend(["GPU Memory (GB)", "Train Time (s)", "Val Time (s)"])
    
    return best_df[display_cols]


best_per_dataset = get_best_per_dataset(merged_df)
if best_per_dataset is not None:
    print("=== Best Configuration per Dataset ===")
    display(best_per_dataset)
    print("\n=== LaTeX Table ===")
    latex_best = generate_latex_table(
        best_per_dataset,
        caption="Best model configuration per dataset with GPU statistics",
        label="tab:best_per_dataset"
    )
    print(latex_best)

In [ ]:
# Cell 8: Best configuration per model (averaged across datasets)

def get_model_summary(df):
    """Get summary statistics per model across all datasets."""
    if df is None or "GPU/peak_memory_allocated_GB_mean" not in df.columns:
        return None
    
    # Filter to rows with GPU stats
    df_valid = df[df["GPU/peak_memory_allocated_GB_mean"].notna()].copy()
    
    if len(df_valid) == 0:
        return None
    
    # Group by model and aggregate
    agg_dict = {
        "summary.best_test/f1_macro": ["mean", "std"],
        "GPU/peak_memory_allocated_GB_mean": ["mean", "std"],
        "AvgTime/train_epoch_mean_mean": ["mean", "std"],
        "AvgTime/val_epoch_mean_mean": ["mean", "std"],
    }
    
    # Add trainable_params if available
    has_params = "trainable_params" in df_valid.columns
    if has_params:
        agg_dict["trainable_params"] = ["mean", "std"]
    
    model_stats = df_valid.groupby("model").agg(agg_dict).round(3)
    
    # Flatten columns
    col_names = [
        "F1 Macro (mean)", "F1 Macro (std)",
        "GPU Memory (mean)", "GPU Memory (std)", 
        "Train Time (mean)", "Train Time (std)",
        "Val Time (mean)", "Val Time (std)"
    ]
    if has_params:
        col_names.extend(["Params (mean)", "Params (std)"])
    model_stats.columns = col_names
    
    model_stats = model_stats.reset_index()
    
    # Format with ±
    model_stats["F1 Macro"] = model_stats.apply(
        lambda r: f"{r['F1 Macro (mean)']:.3f} ± {r['F1 Macro (std)']:.3f}" if pd.notna(r['F1 Macro (std)']) else f"{r['F1 Macro (mean)']:.3f}",
        axis=1
    )
    model_stats["GPU Memory (GB)"] = model_stats.apply(
        lambda r: f"{r['GPU Memory (mean)']:.2f} ± {r['GPU Memory (std)']:.2f}" if pd.notna(r['GPU Memory (std)']) else f"{r['GPU Memory (mean)']:.2f}",
        axis=1
    )
    model_stats["Train Time (s)"] = model_stats.apply(
        lambda r: f"{r['Train Time (mean)']:.2f} ± {r['Train Time (std)']:.2f}" if pd.notna(r['Train Time (std)']) else f"{r['Train Time (mean)']:.2f}",
        axis=1
    )
    model_stats["Val Time (s)"] = model_stats.apply(
        lambda r: f"{r['Val Time (mean)']:.2f} ± {r['Val Time (std)']:.2f}" if pd.notna(r['Val Time (std)']) else f"{r['Val Time (mean)']:.2f}",
        axis=1
    )
    
    # Format params if available
    if has_params:
        def fmt_params_range(row):
            mean = row["Params (mean)"]
            std = row["Params (std)"]
            if pd.isna(mean):
                return "-"
            if mean >= 1e6:
                return f"{mean/1e6:.1f}M"
            elif mean >= 1e3:
                return f"{mean/1e3:.0f}K"
            else:
                return f"{mean:.0f}"
        model_stats["Params"] = model_stats.apply(fmt_params_range, axis=1)
    
    result_cols = ["model", "F1 Macro"]
    if has_params:
        result_cols.append("Params")
    result_cols.extend(["GPU Memory (GB)", "Train Time (s)", "Val Time (s)"])
    
    return model_stats[result_cols].sort_values("model")


model_summary = get_model_summary(merged_df)
if model_summary is not None:
    print("=== Model Summary (averaged across all configurations) ===")
    display(model_summary)
    print("\n=== LaTeX Table ===")
    latex_model = generate_latex_table(
        model_summary,
        caption="Average performance and resource usage per model architecture",
        label="tab:model_summary"
    )
    print(latex_model)
else:
    print("No GPU stats available yet - run the rerun experiments first")

In [ ]:
# Cell 9: Generate Extended LaTeX Table (Best per Dataset-Model with GPU Stats)

def generate_extended_latex_table(df):
    """Generate LaTeX table for best config per (dataset, model) with GPU stats.
    
    Features:
    - Performance metrics: mean ± std, bold for best, purple bg within 1 std of best
    - Efficiency metrics (GPU, Train, Val): mean only (no std), bold for best (minimum)
    - Dataset headers with gray background
    """
    
    # Dataset display name mapping
    dataset_display = {
        "motrpac": "Heritage",
        "addneuromed": "Addneuromed",
        "parkinsons": "Parkinsons",
    }
    
    # Model display name mapping
    model_display = {
        "gatv4": "GATv4",
        "graph_sage": "sage",
        "chebnet": "chebnet",
        "gatv2": "gatv2",
        "gcn": "gcn",
        "gin": "gin",
        "mlp": "mlp",
        "sagn": "sagn",
    }
    
    # Model order for display
    model_order = ["GATv4", "chebnet", "gatv2", "gcn", "gin", "mlp", "sage", "sagn"]
    
    # Dataset order
    dataset_order = ["Heritage", "Addneuromed", "Parkinsons"]
    
    # Performance metric columns (higher is better)
    perf_cols = [
        ("summary.best_test/f1_macro", "summary.best_test/f1_macro_std", "F_macro", 3),
        ("summary.best_test/f1_weighted", "summary.best_test/f1_weighted_std", "F_weighted", 3),
        ("summary.best_test/accuracy", "summary.best_test/accuracy_std", "Accuracy", 3),
        ("summary.best_test/auroc", "summary.best_test/auroc_std", "AUROC", 3),
    ]
    
    # Efficiency metric columns (lower is better) - no std
    eff_cols = [
        ("GPU/peak_memory_allocated_GB_mean", "GPU_Mem", 2),
        ("AvgTime/train_epoch_mean_mean", "Train_Time", 2),
        ("AvgTime/val_epoch_mean_mean", "Val_Time", 2),
    ]
    
    # Filter to rows with GPU stats
    df_valid = df[df["GPU/peak_memory_allocated_GB_mean"].notna()].copy()
    
    if len(df_valid) == 0:
        print("No data with GPU stats available!")
        return None
    
    # Find best config per (dataset, model) by val_f1_macro
    best_idx = df_valid.groupby(["dataset", "model"])["val_f1_macro"].idxmax()
    best_df = df_valid.loc[best_idx].copy()
    
    # Add display names
    best_df["dataset_display"] = best_df["dataset"].map(dataset_display)
    best_df["model_display"] = best_df["model"].map(model_display)
    
    # Check for trainable params
    has_params = "trainable_params" in best_df.columns
    
    # Generate LaTeX
    latex_lines = []
    latex_lines.append(r"\begin{table*}[t]")
    latex_lines.append(r"\centering")
    latex_lines.append(r"\small")
    latex_lines.append(r"\setlength{\tabcolsep}{4pt}")
    latex_lines.append(r"\renewcommand{\arraystretch}{1.15}")
    
    # Column spec
    if has_params:
        latex_lines.append(r"\begin{tabularx}{\textwidth}{l l Y Y Y Y r r r r}")
        ncols = 10
    else:
        latex_lines.append(r"\begin{tabularx}{\textwidth}{l l Y Y Y Y r r r}")
        ncols = 9
    
    latex_lines.append(r"\toprule")
    if has_params:
        latex_lines.append(r"Dataset & Model & $F_{macro}$ & $F_{weighted}$ & Accuracy & AUROC & Params & GPU (GB) & Train (s) & Val (s) \\")
    else:
        latex_lines.append(r"Dataset & Model & $F_{macro}$ & $F_{weighted}$ & Accuracy & AUROC & GPU (GB) & Train (s) & Val (s) \\")
    latex_lines.append(r"\midrule")
    
    for dataset in dataset_order:
        dataset_df = best_df[best_df["dataset_display"] == dataset].copy()
        if len(dataset_df) == 0:
            continue
        
        # Sort by model order
        dataset_df["model_order"] = dataset_df["model_display"].apply(lambda x: model_order.index(x) if x in model_order else 99)
        dataset_df = dataset_df.sort_values("model_order")
        
        # Compute best values for this dataset
        # Performance metrics: max is best
        perf_best = {}
        perf_best_std = {}
        for mean_col, std_col, name, prec in perf_cols:
            perf_best[name] = dataset_df[mean_col].max()
            # Get the std of the best performer
            best_row_idx = dataset_df[mean_col].idxmax()
            perf_best_std[name] = dataset_df.loc[best_row_idx, std_col] if pd.notna(dataset_df.loc[best_row_idx, std_col]) else 0
        
        # Efficiency metrics: min is best
        eff_best = {}
        for mean_col, name, prec in eff_cols:
            eff_best[name] = dataset_df[mean_col].min()
        
        # Params: min is best
        if has_params:
            params_best = dataset_df["trainable_params"].min()
        
        # Dataset header with gray background
        latex_lines.append(f"\\rowcolor{{gray!20}} \\multicolumn{{{ncols}}}{{l}}{{\\textbf{{{dataset}}}}}\\\\")
        
        for _, row in dataset_df.iterrows():
            model = row["model_display"]
            cells = [f"& {model}"]
            
            # Performance metrics (mean ± std, bold best, purple within 1 std)
            for mean_col, std_col, name, prec in perf_cols:
                val = row[mean_col]
                std = row[std_col]
                
                if pd.isna(val):
                    cells.append("-")
                    continue
                
                # Format value
                if pd.isna(std):
                    formatted = f"{val:.{prec}f}"
                else:
                    formatted = f"{val:.{prec}f} $\\pm$ {std:.{prec}f}"
                
                # Check if best (max)
                is_best = abs(val - perf_best[name]) < 1e-9
                # Check if within 1 std of best
                within_std = val >= (perf_best[name] - perf_best_std[name])
                
                if is_best:
                    cells.append(f"\\cellcolor{{gray!30}}\\textbf{{{formatted}}}")
                elif within_std:
                    cells.append(f"\\cellcolor{{violet!15}}{formatted}")
                else:
                    cells.append(formatted)
            
            # Params (if available) - min is best
            if has_params:
                params_val = row["trainable_params"]
                if pd.isna(params_val):
                    cells.append("-")
                else:
                    if params_val >= 1e6:
                        params_fmt = f"{params_val/1e6:.1f}M"
                    elif params_val >= 1e3:
                        params_fmt = f"{params_val/1e3:.0f}K"
                    else:
                        params_fmt = f"{params_val:.0f}"
                    
                    is_best = abs(params_val - params_best) < 1e-9
                    if is_best:
                        cells.append(f"\\cellcolor{{gray!30}}\\textbf{{{params_fmt}}}")
                    else:
                        cells.append(params_fmt)
            
            # Efficiency metrics (mean only, bold min)
            for mean_col, name, prec in eff_cols:
                val = row[mean_col]
                
                if pd.isna(val):
                    cells.append("-")
                    continue
                
                formatted = f"{val:.{prec}f}"
                is_best = abs(val - eff_best[name]) < 1e-9
                
                if is_best:
                    cells.append(f"\\cellcolor{{gray!30}}\\textbf{{{formatted}}}")
                else:
                    cells.append(formatted)
            
            line = " & ".join(cells) + " \\\\"
            latex_lines.append(line)
        
        latex_lines.append(r"\midrule")
    
    # Remove last midrule and add bottomrule
    latex_lines[-1] = r"\bottomrule"
    
    latex_lines.append(r"\end{tabularx}")
    latex_lines.append(r"\caption{Best configuration per model and dataset with GPU memory and timing statistics. Selected by validation $F_{macro}$; test performance shown as mean $\pm$ std. Best values per dataset are bold (gray). Performance metrics within 1 std of best are shaded purple.}")
    latex_lines.append(r"\label{tab:best-configs-with-gpu}")
    latex_lines.append(r"\end{table*}")
    
    return "\n".join(latex_lines)


# Generate the extended LaTeX table
latex_extended = generate_extended_latex_table(merged_df)
if latex_extended:
    print("=== Extended LaTeX Table with GPU Stats ===\n")
    print(latex_extended)

In [ ]:
# Cell 9: Summary Statistics and Insights

def print_summary_stats(df):
    """Print overall summary statistics."""
    if "GPU/peak_memory_allocated_GB_mean" not in df.columns:
        print("No GPU stats available yet")
        return
    
    df_valid = df[df["GPU/peak_memory_allocated_GB_mean"].notna()]
    
    if len(df_valid) == 0:
        print("No configurations with GPU stats available yet")
        return
    
    print("=" * 60)
    print("OVERALL SUMMARY STATISTICS")
    print("=" * 60)
    
    print(f"\nTotal configurations with GPU stats: {len(df_valid)}")
    print(f"Total configurations (all): {len(df)}")
    print(f"Coverage: {len(df_valid)/len(df)*100:.1f}%")
    
    print("\n--- GPU Memory Usage ---")
    gpu_mem = df_valid["GPU/peak_memory_allocated_GB_mean"]
    print(f"  Mean: {gpu_mem.mean():.2f} GB")
    print(f"  Std:  {gpu_mem.std():.2f} GB")
    print(f"  Min:  {gpu_mem.min():.2f} GB ({df_valid.loc[gpu_mem.idxmin(), 'model']})")
    print(f"  Max:  {gpu_mem.max():.2f} GB ({df_valid.loc[gpu_mem.idxmax(), 'model']})")
    
    print("\n--- Training Time (per epoch) ---")
    train_time = df_valid["AvgTime/train_epoch_mean_mean"]
    print(f"  Mean: {train_time.mean():.2f} s")
    print(f"  Std:  {train_time.std():.2f} s")
    print(f"  Min:  {train_time.min():.2f} s ({df_valid.loc[train_time.idxmin(), 'model']})")
    print(f"  Max:  {train_time.max():.2f} s ({df_valid.loc[train_time.idxmax(), 'model']})")
    
    print("\n--- Best F1 Macro ---")
    f1 = df_valid["summary.best_test/f1_macro"]
    best_idx = f1.idxmax()
    best_row = df_valid.loc[best_idx]
    print(f"  Best: {f1.max():.3f}")
    print(f"  Config: {best_row['dataset']} / {best_row['model']} / {best_row['method']} / {best_row['node_sample_ratio']} / {best_row['readout']}")
    
    print("\n" + "=" * 60)


print_summary_stats(merged_df)

In [ ]:
# Cell 10: Export Results

OUTPUT_DIR = Path("../scripts/rerun_best_configs")

# Save merged results (full data)
output_file = OUTPUT_DIR / "merged_performance_gpu_stats.csv"
merged_df.to_csv(output_file, index=False)
print(f"Saved merged results to: {output_file}")

# Save summary table if available
if summary_table is not None:
    summary_file = OUTPUT_DIR / "gpu_stats_summary_table.csv"
    summary_table.to_csv(summary_file, index=False)
    print(f"Saved summary table to: {summary_file}")

# Create and save extended summary with GPU memory, train time, and val time
def create_extended_summary(df):
    """Create extended summary with formatted GPU and time stats."""
    extended = df.copy()
    
    # Add formatted columns for GPU stats and times
    if "GPU/peak_memory_allocated_GB_mean" in extended.columns:
        extended["GPU Memory (GB)"] = extended.apply(
            format_metric_with_std("GPU/peak_memory_allocated_GB_mean", "GPU/peak_memory_allocated_GB_std", 2), axis=1
        )
        extended["Train Time (s)"] = extended.apply(
            format_metric_with_std("AvgTime/train_epoch_mean_mean", "AvgTime/train_epoch_mean_std", 2), axis=1
        )
        extended["Val Time (s)"] = extended.apply(
            format_metric_with_std("AvgTime/val_epoch_mean_mean", "AvgTime/val_epoch_mean_std", 2), axis=1
        )
    
    # Add formatted performance metrics
    extended["F1 Macro"] = extended.apply(
        format_metric_with_std("summary.best_test/f1_macro", "summary.best_test/f1_macro_std", 3), axis=1
    )
    extended["F1 Weighted"] = extended.apply(
        format_metric_with_std("summary.best_test/f1_weighted", "summary.best_test/f1_weighted_std", 3), axis=1
    )
    extended["Accuracy"] = extended.apply(
        format_metric_with_std("summary.best_test/accuracy", "summary.best_test/accuracy_std", 3), axis=1
    )
    extended["AUROC"] = extended.apply(
        format_metric_with_std("summary.best_test/auroc", "summary.best_test/auroc_std", 3), axis=1
    )
    
    # Format trainable params if available
    if "trainable_params" in extended.columns:
        def fmt_params(val):
            if pd.isna(val):
                return "-"
            if val >= 1e6:
                return f"{val/1e6:.1f}M"
            elif val >= 1e3:
                return f"{val/1e3:.0f}K"
            else:
                return f"{val:.0f}"
        extended["Params"] = extended["trainable_params"].apply(fmt_params)
    
    # Select columns for extended summary
    cols = ["dataset", "model", "method", "node_sample_ratio", "readout",
            "F1 Macro", "F1 Weighted", "Accuracy", "AUROC"]
    
    if "Params" in extended.columns:
        cols.append("Params")
    
    if "GPU Memory (GB)" in extended.columns:
        cols.extend(["GPU Memory (GB)", "Train Time (s)", "Val Time (s)"])
    
    return extended[cols].sort_values(["dataset", "model", "method", "node_sample_ratio", "readout"])

extended_summary = create_extended_summary(merged_df)
extended_file = OUTPUT_DIR / "extended_summary_with_gpu_stats.csv"
extended_summary.to_csv(extended_file, index=False)
print(f"Saved extended summary to: {extended_file}")

print(f"\nExtended summary: {len(extended_summary)} rows")
display(extended_summary.head(10))

print("\nDone!")